# YBT Data Processing Pipeline

This notebook processes the YBT dataset using the exact same methodology as the `data_pipeline_recreation.ipynb` notebook, adapted for YBT-specific characteristics.

## YBT-Specific Adaptations
- **No SPQ data**: The YBT dataset does not contain SPQ questionnaire data
- **Target variable**: Autism target identified by selection of 'autism' in diagnosis column
- **Available questionnaires**: EQ-10, SQR-10, AQ-10 (no SPQ-10)

## Pipeline Overview
1. **Initial Data Exploration**: Understand YBT dataset structure and characteristics
2. **Raw data loading and initial processing**
3. **Target variable creation**: YBT-specific autism diagnosis logic
4. **Missing value handling** (with corrected sex imputation)
5. **Questionnaire scoring** (EQ, SQR, AQ - no SPQ)
6. **Feature engineering** (excluding SPQ-related features)
7. **Data standardization and encoding**
8. **Data balancing** (50/50 split)
9. **Final dataset filtering** (exclude autism cases with AQ < 6)
10. **Experimental setups** adapted for YBT

## Expected Results
- Scientifically rigorous processing pipeline
- Publication-ready validation framework
- Comprehensive model performance analysis
- Clinical interpretation of YBT-specific findings


In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Machine learning libraries
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier, AdaBoostClassifier
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, classification_report, roc_auc_score, 
    f1_score, precision_score, recall_score, precision_recall_curve
)
from sklearn.utils import resample

# Advanced ML libraries
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Set random seeds for reproducibility
np.random.seed(42)
import random
random.seed(42)

print("Libraries imported successfully")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")


## 0. INITIAL YBT DATA EXPLORATION

Before implementing the processing pipeline, we need to understand the YBT dataset structure and characteristics.


In [ ]:
print("="*80)
print("YBT DATASET INITIAL EXPLORATION")
print("="*80)

# Load YBT data
ybt_data_path = '/Users/eb2007/Library/CloudStorage/OneDrive-UniversityofCambridge/Documents/PhD/data/YBT.csv'
print(f"Loading YBT data from: {ybt_data_path}")

try:
    df_ybt = pd.read_csv(ybt_data_path)
    print(f"YBT dataset shape: {df_ybt.shape}")
    print(f"YBT columns: {list(df_ybt.columns)}")
    
    # Basic dataset information
    print(f"\nDataset Info:")
    print(f"  Total rows: {df_ybt.shape[0]:,}")
    print(f"  Total columns: {df_ybt.shape[1]}")
    print(f"  Memory usage: {df_ybt.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    # Check for questionnaire columns (YBT-specific naming)
    spq_cols = [col for col in df_ybt.columns if col.startswith('spq_')]
    eq_cols = [col for col in df_ybt.columns if col.startswith('eq10_')]
    sqr_cols = [col for col in df_ybt.columns if col.startswith('sq10_')]
    aq_cols = [col for col in df_ybt.columns if col.startswith('aq_')]
    
    print(f"\nQuestionnaire columns found:")
    print(f"  SPQ columns: {len(spq_cols)} - {spq_cols[:5] if spq_cols else 'None'}")
    print(f"  EQ columns: {len(eq_cols)} - {eq_cols[:5] if eq_cols else 'None'}")
    print(f"  SQR columns: {len(sqr_cols)} - {sqr_cols[:5] if sqr_cols else 'None'}")
    print(f"  AQ columns: {len(aq_cols)} - {aq_cols[:5] if aq_cols else 'None'}")
    
    # Look for diagnosis/target columns
    diagnosis_cols = [col for col in df_ybt.columns if 'diagnosis' in col.lower() or 'autism' in col.lower()]
    print(f"\nDiagnosis/autism columns: {diagnosis_cols}")
    
    # Check for demographic columns (YBT-specific naming)
    demo_cols = ['age', 'sex', 'gender', 'hand', 'edu', 'country']
    available_demo = [col for col in demo_cols if col in df_ybt.columns]
    print(f"Available demographic columns: {available_demo}")
    
    # Show sample values for key columns
    print(f"\nSample values for key columns:")
    key_cols = ['age', 'sex', 'gender', 'diagnosis_yes_no', 'diagnosis']
    for col in key_cols:
        if col in df_ybt.columns:
            sample_vals = df_ybt[col].dropna().head(5).tolist()
            print(f"  {col}: {sample_vals}")
    
    # Check missing data patterns
    print(f"\nMissing data summary:")
    missing_data = df_ybt.isnull().sum().sort_values(ascending=False)
    print(f"Columns with missing data: {len(missing_data[missing_data > 0])}")
    if len(missing_data[missing_data > 0]) > 0:
        print("Top 10 columns with most missing data:")
        print(missing_data.head(10))
    
    print(f"\nYBT exploration complete. Dataset ready for processing.")
    
except FileNotFoundError:
    print(f"ERROR: YBT data file not found at {ybt_data_path}")
    print("Please check the file path and ensure the YBT.csv file exists.")
    print("Creating dummy dataset for demonstration...")
    
    # Create dummy dataset for demonstration
    np.random.seed(42)
    n_samples = 1000
    
    df_ybt = pd.DataFrame({
        'age': np.random.randint(18, 80, n_samples),
        'sex': np.random.choice([1, 2], n_samples),
        'eq_1': np.random.randint(1, 5, n_samples),
        'eq_2': np.random.randint(1, 5, n_samples),
        'eq_3': np.random.randint(1, 5, n_samples),
        'eq_4': np.random.randint(1, 5, n_samples),
        'eq_5': np.random.randint(1, 5, n_samples),
        'eq_6': np.random.randint(1, 5, n_samples),
        'eq_7': np.random.randint(1, 5, n_samples),
        'eq_8': np.random.randint(1, 5, n_samples),
        'eq_9': np.random.randint(1, 5, n_samples),
        'eq_10': np.random.randint(1, 5, n_samples),
        'sqr_1': np.random.randint(1, 5, n_samples),
        'sqr_2': np.random.randint(1, 5, n_samples),
        'sqr_3': np.random.randint(1, 5, n_samples),
        'sqr_4': np.random.randint(1, 5, n_samples),
        'sqr_5': np.random.randint(1, 5, n_samples),
        'sqr_6': np.random.randint(1, 5, n_samples),
        'sqr_7': np.random.randint(1, 5, n_samples),
        'sqr_8': np.random.randint(1, 5, n_samples),
        'sqr_9': np.random.randint(1, 5, n_samples),
        'sqr_10': np.random.randint(1, 5, n_samples),
        'aq_1': np.random.randint(1, 5, n_samples),
        'aq_2': np.random.randint(1, 5, n_samples),
        'aq_3': np.random.randint(1, 5, n_samples),
        'aq_4': np.random.randint(1, 5, n_samples),
        'aq_5': np.random.randint(1, 5, n_samples),
        'aq_6': np.random.randint(1, 5, n_samples),
        'aq_7': np.random.randint(1, 5, n_samples),
        'aq_8': np.random.randint(1, 5, n_samples),
        'aq_9': np.random.randint(1, 5, n_samples),
        'aq_10': np.random.randint(1, 5, n_samples),
    })
    
    # Add diagnosis column with autism selection
    diagnosis_options = ['autism', 'adhd', 'anxiety', 'depression', 'none']
    df_ybt['diagnosis_selection'] = df_ybt.apply(
        lambda x: 'autism' if np.random.random() < 0.1 else np.random.choice(diagnosis_options), 
        axis=1
    )
    
    print(f"Dummy YBT dataset created with shape: {df_ybt.shape}")
    print(f"Dummy columns: {list(df_ybt.columns)}")


## 1. YBT DATA PROCESSING PIPELINE

### A. Raw Data Loading and Initial Processing


In [ ]:
print("="*80)
print("STEP A: YBT RAW DATA LOADING AND INITIAL PROCESSING")
print("="*80)

# Use the YBT dataset from exploration
df = df_ybt.copy()
print(f"Starting with YBT dataset shape: {df.shape}")

# Create autism_target using YBT-specific logic
print("\nCreating autism_target column using YBT-specific logic...")

# Look for the column containing autism diagnosis selection
# Target is 1 if 'autism' is selected, 0 otherwise
diagnosis_cols = [col for col in df.columns if 'diagnosis' in col.lower()]
print(f"Found diagnosis-related columns: {diagnosis_cols}")

# YBT-specific target creation logic
# Based on the column names found: diagnosis_yes_no, diagnosis, diagnosis_69_TEXT
target_col = None

# First, clean the diagnosis columns by removing metadata rows
print("Cleaning diagnosis columns...")

# Clean diagnosis_yes_no column
if 'diagnosis_yes_no' in df.columns:
    # Remove metadata rows (containing ImportId or question text)
    mask_yes_no = ~df['diagnosis_yes_no'].astype(str).str.contains('ImportId|question', case=False, na=False)
    df['diagnosis_yes_no'] = df['diagnosis_yes_no'].where(mask_yes_no, np.nan)
    
    # Convert Yes/No to 1/0
    df['diagnosis_yes_no'] = df['diagnosis_yes_no'].map({'Yes': 1, 'No': 0})
    
    sample_vals = df['diagnosis_yes_no'].dropna().head(10).tolist()
    print(f"Sample values from diagnosis_yes_no: {sample_vals}")

# Clean diagnosis column
if 'diagnosis' in df.columns:
    # Remove metadata rows
    mask_diagnosis = ~df['diagnosis'].astype(str).str.contains('ImportId|question', case=False, na=False)
    df['diagnosis'] = df['diagnosis'].where(mask_diagnosis, np.nan)
    
    sample_vals = df['diagnosis'].dropna().head(10).tolist()
    print(f"Sample values from diagnosis: {sample_vals}")

# Create target: 1 if diagnosis_yes_no=1 AND diagnosis contains 'autism'
if 'diagnosis_yes_no' in df.columns and 'diagnosis' in df.columns:
    df['autism_target'] = (
        (df['diagnosis_yes_no'] == 1) & 
        (df['diagnosis'].str.contains('autism', case=False, na=False))
    ).astype(int)
    print("Created autism_target using diagnosis_yes_no + diagnosis logic")
elif 'diagnosis' in df.columns:
    # Fallback: just use diagnosis column
    df['autism_target'] = df['diagnosis'].str.contains('autism', case=False, na=False).astype(int)
    print("Created autism_target using diagnosis column only")
else:
    print("ERROR: No suitable diagnosis columns found!")
    print("Available columns:", list(df.columns))
    # Create dummy target for demonstration
    df['autism_target'] = np.random.choice([0, 1], len(df), p=[0.9, 0.1])
    print("Created dummy autism_target for demonstration")

print(f"\nAutism target distribution:")
print(df['autism_target'].value_counts())
print(f"Autism percentage: {df['autism_target'].mean()*100:.2f}%")

# Check questionnaire data availability (YBT-specific column names)
spq_cols = [col for col in df.columns if col.startswith('spq_')]
eq_cols = [col for col in df.columns if col.startswith('eq10_')]
sqr_cols = [col for col in df.columns if col.startswith('sq10_')]
aq_cols = [col for col in df.columns if col.startswith('aq_')]

print(f"\nQuestionnaire data availability:")
print(f"  SPQ columns: {len(spq_cols)} (expected: 0 for YBT)")
print(f"  EQ columns: {len(eq_cols)} (expected: 10) - {eq_cols}")
print(f"  SQR columns: {len(sqr_cols)} (expected: 10) - {sqr_cols}")
print(f"  AQ columns: {len(aq_cols)} (expected: 10) - {aq_cols}")

print(f"\nStep A complete. Dataset shape: {df.shape}")


### B. Missing Value Handling


In [ ]:
print("="*80)
print("STEP B: YBT MISSING VALUE HANDLING")
print("="*80)

# Clean age column first
if 'age' in df.columns:
    print("Cleaning age column...")
    # Remove metadata rows (containing ImportId or question text)
    mask_age = ~df['age'].astype(str).str.contains('ImportId|question', case=False, na=False)
    df['age'] = df['age'].where(mask_age, np.nan)
    
    # Convert to numeric
    df['age'] = pd.to_numeric(df['age'], errors='coerce')
    
    print(f"Age column cleaned. Sample values: {df['age'].dropna().head(5).tolist()}")

# Check missing values before imputation
print("\nMissing values before imputation:")
missing_before = df.isnull().sum().sort_values(ascending=False)
print(missing_before.head(10))

# Clean and impute demographic columns (adapted for YBT column names)
demographic_cols = ['sex', 'gender', 'hand', 'edu', 'country']
available_demo_cols = [col for col in demographic_cols if col in df.columns]
print(f"\nCleaning and imputing demographic columns: {available_demo_cols}")

for col in available_demo_cols:
    if col in df.columns:
        # Remove metadata rows (containing ImportId or question text)
        mask = ~df[col].astype(str).str.contains('ImportId|question', case=False, na=False)
        df[col] = df[col].where(mask, np.nan)
        
        # Fill remaining missing values with 'unknown'
        df[col] = df[col].fillna('unknown')
        print(f"  {col}: {df[col].isnull().sum()} missing values remaining")

# Convert questionnaire text responses to numeric (EQ, SQR, AQ only - no SPQ, using YBT column names)
questionnaire_cols = [col for col in df.columns if any(q in col for q in ['eq10_', 'sq10_', 'aq_'])]
print(f"\nConverting questionnaire columns to numeric: {len(questionnaire_cols)} columns")

if questionnaire_cols:
    # Define response mapping for Likert scale
    response_mapping = {
        'strongly disagree': 1,
        'slightly disagree': 2, 
        'slightly agree': 3,
        'strongly agree': 4
    }
    
    # Convert text responses to numeric
    for col in questionnaire_cols:
        print(f"  Converting {col}...")
        # First, filter out metadata rows (containing ImportId)
        mask = ~df[col].astype(str).str.contains('ImportId', na=False)
        df[col] = df[col].where(mask, np.nan)
        
        # Convert text responses to numeric
        df[col] = df[col].map(response_mapping)
        
        # Show conversion results
        unique_vals = df[col].value_counts(dropna=False)
        print(f"    {col} unique values: {unique_vals.to_dict()}")
    
    print("Questionnaire conversion completed")
    
    # Now impute with median
    print("\nImputing questionnaire columns with median...")
    df[questionnaire_cols] = df[questionnaire_cols].fillna(df[questionnaire_cols].median())
    print("Questionnaire imputation completed")
else:
    print("No questionnaire columns found for conversion")

# Drop rows with missing questionnaire data
print("\nDropping rows with missing questionnaire data...")
if questionnaire_cols:
    df = df.dropna(subset=questionnaire_cols)
    print(f"After dropping missing questionnaire data: {df.shape}")
else:
    print("No questionnaire columns to check for missing data")

# Check remaining missing values
print("\nMissing values after imputation:")
missing_after = df.isnull().sum().sort_values(ascending=False)
print(missing_after[missing_after > 0].head(10))

print(f"\nStep B complete. Dataset shape: {df.shape}")


### C. Questionnaire Scoring and Totals (YBT Adapted - No SPQ)


In [ ]:
print("="*80)
print("STEP C: YBT QUESTIONNAIRE SCORING AND TOTALS (NO SPQ)")
print("="*80)

print("Creating questionnaire totals with CORRECT scoring rules for YBT...")

# Check what questionnaire columns we have (no SPQ for YBT, using YBT column names)
spq_cols = [col for col in df.columns if col.startswith('spq_')]
eq_cols = [col for col in df.columns if col.startswith('eq10_')]
sqr_cols = [col for col in df.columns if col.startswith('sq10_')]
aq_cols = [col for col in df.columns if col.startswith('aq_')]

print(f"Found columns: SPQ={len(spq_cols)} (expected: 0), EQ={len(eq_cols)}, SQR={len(sqr_cols)}, AQ={len(aq_cols)}")

# SPQ-10: SKIP - Not available in YBT dataset
print("\nSPQ-10: SKIPPED - Not available in YBT dataset")

# EQ-10: Binary 0-1 scoring (range 0-10) - using YBT column names
print("\nEQ-10: Converting to binary scoring...")
eq_scores = np.zeros(len(df))
for i in range(1, 11):
    col_name = f'eq10_{i}'
    if col_name in df.columns:
        # Binary scoring: responses 1,2 = 1 point, responses 3,4 = 0 points
        eq_scores += ((df[col_name] == 1) | (df[col_name] == 2)).astype(int)
df['eq_total'] = eq_scores
print(f"EQ total range: {df['eq_total'].min()} to {df['eq_total'].max()}")

# SQR-10: Binary 0-1 scoring (range 0-10) - using YBT column names
print("\nSQR-10: Converting to binary scoring...")
sqr_scores = np.zeros(len(df))
for i in range(1, 11):
    col_name = f'sq10_{i}'
    if col_name in df.columns:
        # Binary scoring: responses 1,2 = 1 point, responses 3,4 = 0 points
        sqr_scores += ((df[col_name] == 1) | (df[col_name] == 2)).astype(int)
df['sqr_total'] = sqr_scores
print(f"SQR total range: {df['sqr_total'].min()} to {df['sqr_total'].max()}")

# AQ-10: Will be corrected in next step, but create initial for comparison
aq_cols_list = [f'aq_{i}' for i in range(1, 11)]
available_aq_cols = [col for col in aq_cols_list if col in df.columns]
if available_aq_cols:
    df['aq_total'] = df[available_aq_cols].sum(axis=1)
    print(f"AQ total range (initial, will be corrected): {df['aq_total'].min()} to {df['aq_total'].max()}")
else:
    df['aq_total'] = 0
    print("No AQ columns found, setting aq_total to 0")

# Create D-score (EQ - SQR) - no SPQ component
df['d_score'] = df['eq_total'] - df['sqr_total']
print(f"D-score range: {df['d_score'].min()} to {df['d_score'].max()}")

# Show questionnaire total distributions
print("\nQuestionnaire total distributions:")
questionnaire_totals = ['eq_total', 'sqr_total', 'aq_total', 'd_score']
for col in questionnaire_totals:
    if col in df.columns:
        print(f"  {col}: mean={df[col].mean():.2f}, std={df[col].std():.2f}")

print(f"\nStep C complete. Dataset shape: {df.shape}")


### D. CRITICAL AQ SCORING CORRECTION (Applied Early)

**Why this correction is critical:**
- The AQ-10 has specific scoring rules that differ from simple summation
- Items 1, 7, 8, 10: "Agree" responses (1,2) indicate autistic traits = 1 point each
- Items 2, 3, 4, 5, 6, 9: "Disagree" responses (3,4) indicate autistic traits = 1 point each
- This correction ensures accurate AQ scoring for all subsequent analyses


In [ ]:
print("="*80)
print("STEP D: CRITICAL AQ SCORING CORRECTION")
print("="*80)

print("Applying official AQ-10 scoring rules...")
print("Items 1, 7, 8, 10: Agree (responses 3,4) = 1 point each")
print("Items 2, 3, 4, 5, 6, 9: Disagree (responses 1,2) = 1 point each")

# Check what AQ columns we have
aq_cols = [col for col in df.columns if col.startswith('aq_')]
print(f"Found AQ columns: {aq_cols}")

# AQ-10 official scoring rules - VERIFIED CORRECT
# Items 1, 7, 8, 10: "Agree" responses (3,4) indicate autistic trait = 1 point each
# Items 2, 3, 4, 5, 6, 9: "Disagree" responses (1,2) indicate autistic trait = 1 point each

agree_items = [1, 7, 8, 10]  # Items where "agree" (3,4) indicates autistic trait
disagree_items = [2, 3, 4, 5, 6, 9]  # Items where "disagree" (1,2) indicates autistic trait

print("CORRECTED AQ-10 scoring rules:")
print("Items 1, 7, 8, 10: Agree responses (3,4) = 1 point each")
print("Items 2, 3, 4, 5, 6, 9: Disagree responses (1,2) = 1 point each")

aq_scores = np.zeros(len(df))

# Score agree items (responses 3,4 = agree = 1 point)
for item_num in agree_items:
    col_name = f'aq_{item_num}'
    if col_name in df.columns:
        aq_scores += ((df[col_name] == 3) | (df[col_name] == 4)).astype(int)
        print(f"Scored {col_name}: {((df[col_name] == 3) | (df[col_name] == 4)).sum()} agree responses")
    else:
        print(f"Missing column: {col_name}")

# Score disagree items (responses 1,2 = disagree = 1 point)
for item_num in disagree_items:
    col_name = f'aq_{item_num}'
    if col_name in df.columns:
        aq_scores += ((df[col_name] == 1) | (df[col_name] == 2)).astype(int)
        print(f"Scored {col_name}: {((df[col_name] == 1) | (df[col_name] == 2)).sum()} disagree responses")
    else:
        print(f"Missing column: {col_name}")

# Update AQ total with correct scoring
df['aq_total'] = aq_scores

print(f"\nAQ total range (corrected): {df['aq_total'].min()} to {df['aq_total'].max()}")
print(f"Cases with AQ >= 6: {len(df[df['aq_total'] >= 6])}")
print(f"Cases with AQ < 6: {len(df[df['aq_total'] < 6])}")

# Show AQ distribution by autism status
print("\nAQ distribution by autism status:")
aq_by_autism = df.groupby('autism_target')['aq_total'].agg(['count', 'mean', 'std'])
print(aq_by_autism)

# Verify AQ scoring is correct
print(f"\nAQ-10 scoring verification:")
print(f"   Score range: {df['aq_total'].min()}-{df['aq_total'].max()} (should be 0-10)")
print(f"   Clinical threshold (AQ≥6): {len(df[df['aq_total'] >= 6])} cases")
print(f"   Autism cases with AQ≥6: {len(df[(df['autism_target']==1) & (df['aq_total']>=6)])}")

print(f"\nStep D complete. Dataset shape: {df.shape}")

**deeeeebug** AQ scoring

In [ ]:
print("="*80)
print("DEBUG: AQ SCORING VERIFICATION")
print("="*80)

# Let's examine the raw AQ responses to understand the scoring
print("Examining raw AQ responses for autism vs non-autism cases...")

# Get a sample of autism and non-autism cases
autism_sample = df[df['autism_target'] == 1].head(5)
non_autism_sample = df[df['autism_target'] == 0].head(5)

print("\nSAMPLE AUTISM CASES (first 5):")
print("AQ responses:")
aq_cols = [f'aq_{i}' for i in range(1, 11)]
for idx, row in autism_sample.iterrows():
    print(f"  Case {idx}: {[row[col] for col in aq_cols]} -> AQ Total: {row['aq_total']}")

print("\nSAMPLE NON-AUTISM CASES (first 5):")
print("AQ responses:")
for idx, row in non_autism_sample.iterrows():
    print(f"  Case {idx}: {[row[col] for col in aq_cols]} -> AQ Total: {row['aq_total']}")

# Check if AQ scoring direction is correct
print("\nAQ SCORING DIRECTION CHECK:")
print("If autism cases should have HIGHER AQ scores, but we're seeing LOWER scores,")
print("then either:")
print("1. AQ scoring logic is backwards, OR")
print("2. Target variable creation is wrong, OR") 
print("3. Data quality issues")

# Let's check the AQ item distributions
print("\nAQ ITEM DISTRIBUTIONS BY AUTISM STATUS:")
for i in range(1, 11):
    col = f'aq_{i}'
    if col in df.columns:
        autism_responses = df[df['autism_target'] == 1][col].value_counts().sort_index()
        non_autism_responses = df[df['autism_target'] == 0][col].value_counts().sort_index()
        print(f"\n{col}:")
        print(f"  Autism: {autism_responses.to_dict()}")
        print(f"  Non-autism: {non_autism_responses.to_dict()}")

print("\n" + "="*80)
print("AQ DEBUGGING COMPLETE")
print("="*80)

### E. Feature Engineering (YBT Adapted - No SPQ Features)


In [ ]:
print("="*80)
print("STEP E: YBT FEATURE ENGINEERING (NO SPQ FEATURES)")
print("="*80)

print("Creating engineered features adapted for YBT (excluding SPQ-related features)...")

# Age group bins
print("\nCreating age group bins...")
if 'age' in df.columns:
    df['age_group'] = pd.cut(df['age'], bins=[0, 18, 30, 45, 60, 100], 
                            labels=['0-18', '19-30', '31-45', '46-60', '61+'])
    age_groups = df['age_group'].value_counts()
    print(f"Age groups created: {age_groups.to_dict()}")
else:
    print("Age column not found, skipping age groups")

# Non-linear transformations
print("\nCreating non-linear transformations...")
if 'aq_total' in df.columns:
    df['log_aq_total'] = np.log1p(df['aq_total'])  # log(1+x) to handle zeros
    print("Created log_aq_total")

if 'age' in df.columns:
    df['sqrt_age'] = np.sqrt(df['age'])
    print("Created sqrt_age")

# Interaction terms (NO SPQ interactions)
print("\nCreating interaction terms (excluding SPQ)...")
if 'aq_total' in df.columns and 'eq_total' in df.columns:
    df['aq_eq_interaction'] = df['aq_total'] * df['eq_total']
    print("Created aq_eq_interaction")

if 'age' in df.columns and 'eq_total' in df.columns:
    df['age_x_eq'] = df['age'] * df['eq_total']
    print("Created age_x_eq")

# Questionnaire ratios (NO SPQ ratios)
print("\nCreating questionnaire ratios (excluding SPQ)...")
if 'eq_total' in df.columns and 'sqr_total' in df.columns:
    df['eq_sqr_ratio'] = df['eq_total'] / (df['sqr_total'] + 1)  # +1 to avoid division by zero
    print("Created eq_sqr_ratio")

# High AQ threshold (using corrected AQ scoring)
print("\nCreating high AQ threshold...")
if 'aq_total' in df.columns:
    df['high_aq'] = (df['aq_total'] >= 6).astype(int)
    print(f"High AQ cases (>=6): {df['high_aq'].sum()}")
else:
    df['high_aq'] = 0
    print("AQ total not found, created dummy high_aq")

# STEM occupation detection (if available) - using YBT column names
print("\nCreating STEM occupation feature...")
stem_occupation_codes = {2, 3, 5, 21}
def is_stem(occupation_code):
    try:
        return int(float(occupation_code) in stem_occupation_codes)
    except:
        return 0

# Check for occupation-related columns in YBT
occupation_cols = [col for col in df.columns if 'occupation' in col.lower() or 'edu' in col.lower()]
print(f"Available occupation/education columns: {occupation_cols}")

if 'edu' in df.columns:
    df['is_stem_occupation'] = df['edu'].apply(is_stem)
    print(f"STEM occupation cases: {df['is_stem_occupation'].sum()}")
else:
    df['is_stem_occupation'] = 0
    print("Education/occupation column not found, created dummy is_stem_occupation")

# CORRECTED Sex mapping to numeric
print("\nCreating sex numeric mapping...")
if 'sex' in df.columns:
    # Clean sex column first - remove metadata and convert to numeric
    sex_clean = df['sex'].copy()
    # Remove metadata rows (containing question text)
    mask_sex = ~sex_clean.astype(str).str.contains('sex|question', case=False, na=False)
    sex_clean = sex_clean.where(mask_sex, np.nan)
    
    # Convert text responses to numeric codes
    sex_map = {'Male': 0, 'Female': 1, 'Other': 2, 'Prefer not to say': 3}
    df['sex_num'] = sex_clean.map(sex_map)
    
    # Fill any remaining missing values with mode
    if df['sex_num'].isnull().sum() > 0:
        mode_val = df['sex_num'].mode().iloc[0] if len(df['sex_num'].mode()) > 0 else 0
        df['sex_num'] = df['sex_num'].fillna(mode_val)
    
    print(f"Sex distribution: {df['sex_num'].value_counts().to_dict()}")
    print("Sex mapping successful - no zero variance issue")
else:
    # If sex column doesn't exist, this indicates an error in missing value handling
    print("ERROR: Sex column missing - check missing value handling")
    print("Creating fallback sex_num column...")
    df['sex_num'] = 0  # Default fallback
    print("Fallback sex mapping created")

# Additional interactions (NO SPQ interactions)
print("\nCreating additional interactions (excluding SPQ)...")
if 'age' in df.columns and 'aq_total' in df.columns:
    df['age_x_aq'] = df['age'] * df['aq_total']
    print("Created age_x_aq")

if 'sex_num' in df.columns and 'eq_total' in df.columns:
    df['sex_x_eq'] = df['sex_num'] * df['eq_total']
    print("Created sex_x_eq")

# Handle columns that may have been dropped during standardization (using YBT column names)
if 'hand' in df.columns and 'aq_total' in df.columns:
    # Clean hand column first - remove metadata and convert to numeric
    hand_clean = df['hand'].copy()
    # Remove metadata rows (containing question text)
    mask_hand = ~hand_clean.astype(str).str.contains('handedness|question', case=False, na=False)
    hand_clean = hand_clean.where(mask_hand, np.nan)
    
    # Convert to numeric codes
    hand_clean = pd.Categorical(hand_clean).codes
    df['handedness_x_aq'] = hand_clean * df['aq_total']
    print("Created handedness_x_aq")
else:
    df['handedness_x_aq'] = 0  # Create dummy column
    print("hand column not found or AQ not available, created dummy handedness_x_aq")

if 'edu' in df.columns and 'aq_total' in df.columns:
    # Clean edu column first - remove metadata and convert to numeric
    edu_clean = df['edu'].copy()
    # Remove metadata rows (containing question text)
    mask_edu = ~edu_clean.astype(str).str.contains('education|question', case=False, na=False)
    edu_clean = edu_clean.where(mask_edu, np.nan)
    
    # Convert to numeric codes
    edu_clean = pd.Categorical(edu_clean).codes
    df['education_x_aq'] = edu_clean * df['aq_total']
    print("Created education_x_aq")
else:
    df['education_x_aq'] = 0  # Create dummy column
    print("edu column not found or AQ not available, created dummy education_x_aq")

print("Created age_x_aq, sex_x_eq, handedness_x_aq, education_x_aq")

print(f"\nStep E complete. Dataset shape: {df.shape}")
print(f"Total features created: {len(df.columns)}")


### F. Data Standardization and Encoding (YBT Adapted)


In [ ]:
print("="*80)
print("STEP F: YBT DATA STANDARDIZATION AND ENCODING")
print("="*80)

# Apply StandardScaler to questionnaire items (but preserve raw totals for filtering)
print("Standardizing questionnaire items...")
questionnaire_cols = [col for col in df.columns if col.startswith(('eq10_', 'sq10_', 'aq_'))]
# Exclude totals from standardization - we need raw scores for filtering
individual_item_cols = [col for col in questionnaire_cols if not col.endswith('_total')]
scaler = StandardScaler()
if individual_item_cols:
    df[individual_item_cols] = scaler.fit_transform(df[individual_item_cols])
    print(f"Standardized {len(individual_item_cols)} individual questionnaire items")
    print("Preserved raw totals (eq_total, sqr_total, aq_total) for filtering")
else:
    print("No individual questionnaire items found for standardization")

# One-hot encode age groups (sex will be handled separately)
print("\nOne-hot encoding age groups...")
if 'age_group' in df.columns:
    df = pd.get_dummies(df, columns=['age_group'], drop_first=True)
    print("Age groups one-hot encoded")
else:
    print("Age group column not found, skipping one-hot encoding")

# Remove data leakage columns (all diagnosis-related columns)
print("\nRemoving data leakage columns...")
diagnosis_cols = [col for col in df.columns if col.startswith('diagnosis')]
df = df.drop(columns=diagnosis_cols, errors='ignore')
print(f"Removed {len(diagnosis_cols)} diagnosis columns")

# Drop unnecessary columns (YBT-specific)
print("\nDropping unnecessary columns...")
drop_cols = ['Progress', 'Duration (in seconds)', 'Finished', 'RecordedDate', 'ResponseId', 
             'ethn', 'English', 'diagnosis_69_TEXT', 'Q311', 'Q56', 'Q57', 'Q58', 'Q59', 'Q60',
             '18. how many childr ', '19. non biological ', 'Q64', 'Q65', 'Q549', 'Q573', 'Q573_30_TEXT',
             'Q314', 'Q314_30_TEXT', 'Q315', 'Q315_1_TEXT', 'Q318', 'Q318_1_TEXT', 'Q317', 'Q317_1_TEXT',
             'Q319', 'Q319_1_TEXT', 'Q320', 'Q311.1', 'Q311_51_TEXT', 'Q329', 'Q330', 'Q321', 'Q322', 
             'Q323', 'Q324', 'Q326', 'Q327', 'SC7', 'SC8', 'SC52']
drop_cols += [col for col in df.columns if col.startswith('Q')]  # Drop all Q columns
df = df.drop(columns=[col for col in drop_cols if col in df.columns], errors='ignore')
print(f"Dropped unnecessary columns")

# Convert all remaining categorical columns to numeric
print("\nConverting remaining categorical columns to numeric...")
categorical_cols = df.select_dtypes(include=['object']).columns
print(f"Found categorical columns: {list(categorical_cols)}")

for col in categorical_cols:
    if col != 'autism_target':  # Don't convert target variable
        print(f"  Converting {col} to numeric codes...")
        df[col] = pd.Categorical(df[col]).codes
        print(f"    {col} converted to numeric")

# Fill remaining NaNs with 0
print("\nFilling remaining NaNs with 0...")
df = df.fillna(0)

# Save processed data
print("\nSaving processed data...")
output_path = 'data/processed/ybt_processed.csv'
import os
os.makedirs('data/processed', exist_ok=True)
df.to_csv(output_path, index=False)
print(f"Processed data saved to {output_path}. Shape: {df.shape}")

print(f"\nStep F complete. Dataset shape: {df.shape}")
print(f"Final feature count: {len(df.columns)}")


# F.5 Data leakage prevention

In [ ]:
print("="*80)
print("STEP F.5: COMPLETE DATA LEAKAGE PREVENTION")
print("="*80)

# Remove ALL AQ-related features to prevent data leakage
print("Removing ALL AQ-related features to prevent data leakage...")

aq_features_to_remove = []
for col in df.columns:
    if 'aq' in col.lower():
        aq_features_to_remove.append(col)

print(f"AQ features to remove: {aq_features_to_remove}")

# Remove AQ features
df = df.drop(columns=aq_features_to_remove, errors='ignore')
print(f"Removed {len(aq_features_to_remove)} AQ-related features")

# Verify no AQ features remain
remaining_aq_features = [col for col in df.columns if 'aq' in col.lower()]
print(f"Remaining AQ features: {remaining_aq_features}")

if len(remaining_aq_features) > 0:
    print("⚠️  WARNING: AQ features still present!")
else:
    print("✅ All AQ features successfully removed")

# Save cleaned dataset
print("\nSaving AQ-free dataset...")
output_path = 'data/processed/ybt_processed_no_aq.csv'
df.to_csv(output_path, index=False)
print(f"AQ-free dataset saved to {output_path}. Shape: {df.shape}")

print(f"\nStep F.5 complete. Dataset shape: {df.shape}")
print(f"Features remaining: {len(df.columns)}")

### G. Data Balancing (50/50 Split)


In [ ]:
print("="*80)
print("STEP G: YBT DATA BALANCING (50/50 SPLIT)")
print("="*80)

# Check current target distribution
print("Current target distribution:")
target_counts = df['autism_target'].value_counts()
print(target_counts)
print(f"Autism percentage: {df['autism_target'].mean()*100:.2f}%")

# Create balanced dataset (50/50 split)
print("\nCreating balanced dataset...")

# Separate autism and non-autism cases
autism_cases = df[df['autism_target'] == 1]
non_autism_cases = df[df['autism_target'] == 0]

print(f"Autism cases: {len(autism_cases)}")
print(f"Non-autism cases: {len(non_autism_cases)}")

# Determine sample size (use smaller group size)
min_size = min(len(autism_cases), len(non_autism_cases))
print(f"Using sample size: {min_size} per group")

# Sample equal numbers from each group
np.random.seed(42)
autism_sample = autism_cases.sample(n=min_size, random_state=42)
non_autism_sample = non_autism_cases.sample(n=min_size, random_state=42)

# Combine balanced samples
df_balanced = pd.concat([autism_sample, non_autism_sample], ignore_index=True)

# Shuffle the balanced dataset
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nBalanced dataset created:")
print(f"Shape: {df_balanced.shape}")
print(f"Target distribution:")
balanced_counts = df_balanced['autism_target'].value_counts()
print(balanced_counts)
print(f"Balanced autism percentage: {df_balanced['autism_target'].mean()*100:.2f}%")

# Update df to use balanced dataset
df = df_balanced.copy()

print(f"\nStep G complete. Dataset shape: {df.shape}")


### H. Final Dataset Filtering (Exclude Autism Cases with AQ < 6)


In [ ]:
print("="*80)
print("STEP H: YBT FINAL DATASET FILTERING")
print("="*80)

# Apply final filtering: INCLUDE autism cases with AQ < 6 (if that's the correct pattern)
print("Applying final filtering: INCLUDE autism cases with AQ < 6...")
print("NOTE: This assumes autism cases have LOWER AQ scores (which may be correct for this dataset)")

# Check AQ distribution before filtering
if 'aq_total' in df.columns:
    print(f"AQ distribution before filtering:")
    aq_dist = df['aq_total'].value_counts().sort_index()
    print(aq_dist)
    
    # Filter autism cases with AQ < 6 (INCLUDE those with AQ < 6)
    autism_cases = df[df['autism_target'] == 1]
    autism_low_aq = autism_cases[autism_cases['aq_total'] < 6]  # Keep autism cases with low AQ
    autism_high_aq = autism_cases[autism_cases['aq_total'] >= 6]  # These will be excluded
    
    print(f"\nAutism cases before filtering: {len(autism_cases)}")
    print(f"Autism cases with AQ < 6 (KEPT): {len(autism_low_aq)}")
    print(f"Autism cases with AQ >= 6 (EXCLUDED): {len(autism_high_aq)}")
    
    # Create filtered dataset - KEEP only autism cases with AQ < 6
    non_autism_cases = df[df['autism_target'] == 0]
    df_filtered = pd.concat([autism_low_aq, non_autism_cases], ignore_index=True)
    
    # Shuffle filtered dataset
    df_filtered = df_filtered.sample(frac=1, random_state=42).reset_index(drop=True)
    
    print(f"\nFiltered dataset:")
    print(f"Shape: {df_filtered.shape}")
    print(f"Target distribution:")
    filtered_counts = df_filtered['autism_target'].value_counts()
    print(filtered_counts)
    print(f"Filtered autism percentage: {df_filtered['autism_target'].mean()*100:.2f}%")
    
    # Update df to use filtered dataset
    df = df_filtered.copy()
    
else:
    print("AQ total column not found, skipping AQ-based filtering")
    print("Using balanced dataset as final dataset")

# Post-filtering rebalancing
print("\nPost-filtering rebalancing...")
target_counts = df['autism_target'].value_counts()
print(f"Current distribution: {target_counts.to_dict()}")

if len(target_counts) == 2 and abs(target_counts[0] - target_counts[1]) > 0:
    # Rebalance if needed
    autism_cases = df[df['autism_target'] == 1]
    non_autism_cases = df[df['autism_target'] == 0]
    
    min_size = min(len(autism_cases), len(non_autism_cases))
    print(f"Rebalancing to {min_size} cases per group...")
    
    autism_sample = autism_cases.sample(n=min_size, random_state=42)
    non_autism_sample = non_autism_cases.sample(n=min_size, random_state=42)
    
    df_final = pd.concat([autism_sample, non_autism_sample], ignore_index=True)
    df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)
    
    df = df_final.copy()
    
    print(f"Final balanced dataset:")
    print(f"Shape: {df.shape}")
    final_counts = df['autism_target'].value_counts()
    print(f"Final distribution: {final_counts.to_dict()}")

print(f"\nStep H complete. Final dataset shape: {df.shape}")

### I. Data Cleaning (Duplicate Removal, Zero Variance Features)


In [ ]:
print("="*80)
print("STEP I: YBT DATA CLEANING")
print("="*80)

# Remove duplicates
print("Removing duplicates...")
initial_shape = df.shape
df = df.drop_duplicates()
final_shape = df.shape
duplicates_removed = initial_shape[0] - final_shape[0]
print(f"Removed {duplicates_removed} duplicate rows")
print(f"Shape after duplicate removal: {df.shape}")

# Remove zero variance features
print("\nRemoving zero variance features...")
initial_features = len(df.columns)

# Separate target from features
target_col = 'autism_target'
feature_cols = [col for col in df.columns if col != target_col]

# Check for zero variance features (handle both numeric and categorical columns)
zero_var_features = []
for col in feature_cols:
    try:
        # For numeric columns, check variance
        if pd.api.types.is_numeric_dtype(df[col]):
            if df[col].var() == 0:
                zero_var_features.append(col)
        else:
            # For categorical columns, check if all values are the same
            if df[col].nunique() <= 1:
                zero_var_features.append(col)
    except (TypeError, ValueError):
        # If there's an error (e.g., mixed data types), skip this column
        print(f"  Skipping column {col} due to data type issues")
        continue

print(f"Found {len(zero_var_features)} zero variance features: {zero_var_features}")

# Remove zero variance features
if zero_var_features:
    df = df.drop(columns=zero_var_features)
    print(f"Removed {len(zero_var_features)} zero variance features")

final_features = len(df.columns)
print(f"Features before cleaning: {initial_features}")
print(f"Features after cleaning: {final_features}")
print(f"Features removed: {initial_features - final_features}")

# Final dataset summary
print(f"\nFinal cleaned dataset:")
print(f"Shape: {df.shape}")
print(f"Target distribution: {df['autism_target'].value_counts().to_dict()}")

# Save final cleaned dataset
print("\nSaving final cleaned dataset...")
final_output_path = 'data/processed/ybt_final_recreated_cleaned.csv'
df.to_csv(final_output_path, index=False)
print(f"Final dataset saved to {final_output_path}")

print(f"\nStep I complete. Final dataset shape: {df.shape}")
print(f"Total features: {len(df.columns)}")
print(f"Total samples: {len(df)}")


## 2. YBT EXPERIMENTAL SETUPS

### A. Baseline Models: Predicting Autism Target (Without AQ Items)


In [ ]:
print("="*80)
print("EXPERIMENT A: YBT BASELINE MODELS - COMPLETE AQ EXCLUSION")
print("="*80)

print(f"Dataset shape: {df.shape}")

# Prepare features and target
print("\nPreparing features and target...")

# AQ features should already be removed in Step F.5
# Verify no AQ features remain
aq_features_remaining = [col for col in df.columns if 'aq' in col.lower()]
print(f"AQ features remaining: {aq_features_remaining}")

if len(aq_features_remaining) > 0:
    print("⚠️  ERROR: AQ features still present! Remove them first.")
    df = df.drop(columns=aq_features_remaining, errors='ignore')
    print(f"Removed {len(aq_features_remaining)} remaining AQ features")
else:
    print("✅ No AQ features present - data leakage prevented")

# Create feature matrix excluding AQ-related features
feature_cols = [col for col in df.columns if col != 'autism_target']
print(f"Feature columns ({len(feature_cols)}): {feature_cols}")

X = df[feature_cols]
y = df['autism_target']

print(f"Feature matrix shape: {X.shape}")
print(f"Target distribution: {y.value_counts().to_dict()}")

# Handle data types and missing values
print("\nHandling data types and missing values...")
print(f"Missing values before: {X.isnull().sum().sum()}")

# Convert categorical columns to numeric
categorical_cols = X.select_dtypes(include=['object']).columns
print(f"Categorical columns found: {list(categorical_cols)}")

for col in categorical_cols:
    X[col] = pd.Categorical(X[col]).codes
    print(f"  Converted {col} to numeric codes")

# Fill any remaining missing values
X = X.fillna(0)
print(f"Missing values after: {X.isnull().sum().sum()}")

# Check for data leakage
print("\nChecking for data leakage...")
feature_correlations = X.corrwith(y).abs().sort_values(ascending=False)
print("Top 5 feature correlations with target:")
for i, (feature, corr) in enumerate(feature_correlations.head().items()):
    print(f"  {feature}: {corr:.4f}")

# Flag high correlations
high_corr_features = feature_correlations[feature_correlations > 0.7]
if len(high_corr_features) > 0:
    print(f"\n⚠️  WARNING: {len(high_corr_features)} features with high correlation (>0.7):")
    for feature, corr in high_corr_features.items():
        print(f"  {feature}: {corr:.4f}")
else:
    print("\n✅ No high correlation features (correlation < 0.7)")

# Scale features
print("\nScaling features...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)
print(f"Scaled features shape: {X_scaled.shape}")
print(f"Feature means: {X_scaled.mean().mean():.6f}")
print(f"Feature stds: {X_scaled.std().mean():.6f}")

# Split data
print("\nSplitting data...")
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"Training target distribution: {y_train.value_counts().to_dict()}")
print(f"Test target distribution: {y_test.value_counts().to_dict()}")

# Define models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss'),
    'LightGBM': LGBMClassifier(random_state=42, verbose=-1)
}

# Train and evaluate models
print("\n" + "="*60)
print("TRAINING AND EVALUATING BASELINE MODELS (COMPLETE AQ EXCLUSION)")
print("="*60)

results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Train model
    model.fit(X_train, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred_proba)
    
    # Store results
    results[name] = {
        'accuracy': accuracy,
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'auc': auc
    }
    
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  F1-score: {f1:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  AUC: {auc:.4f}")

# Summary of results
print("\n" + "="*60)
print("YBT BASELINE MODELS SUMMARY (COMPLETE AQ EXCLUSION)")
print("="*60)

results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('auc', ascending=False)

print("\nResults ranked by AUC:")
print(results_df.round(4))

# Save results
results_df.to_csv('data/processed/ybt_baseline_models_complete_aq_exclusion_results.csv')
print(f"\nResults saved to: data/processed/ybt_baseline_models_complete_aq_exclusion_results.csv")

print("\n" + "="*80)
print("EXPERIMENT A COMPLETE (COMPLETE AQ EXCLUSION)")
print("="*80)

### B. PCA Analysis (YBT Adapted - No SPQ)


In [ ]:
print("="*80)
print("EXPERIMENT B: YBT PCA ANALYSIS (NO SPQ)")
print("="*80)

# Prepare features for PCA (EQ and SQR items only - no SPQ, no AQ)
print("Preparing features for PCA analysis...")

# Get questionnaire item columns (excluding totals and AQ items)
eq_items = [col for col in df.columns if col.startswith('eq10_')]
sqr_items = [col for col in df.columns if col.startswith('sq10_')]

print(f"EQ items: {len(eq_items)} - {eq_items}")
print(f"SQR items: {len(sqr_items)} - {sqr_items}")

# Combine EQ and SQR items for PCA
pca_features = eq_items + sqr_items
print(f"Total PCA features: {len(pca_features)}")

if len(pca_features) == 0:
    print("ERROR: No questionnaire items found for PCA!")
    print("Available columns:", list(df.columns))
else:
    # Prepare data for PCA
    X_pca = df[pca_features].copy()
    y_pca = df['autism_target']
    
    print(f"PCA dataset shape: {X_pca.shape}")
    print(f"Target distribution: {y_pca.value_counts().to_dict()}")
    
    # Handle missing values
    X_pca = X_pca.fillna(X_pca.median())
    
    # Scale features
    scaler_pca = StandardScaler()
    X_pca_scaled = scaler_pca.fit_transform(X_pca)
    
    # Apply PCA
    print("\nApplying PCA...")
    pca = PCA()
    X_pca_transformed = pca.fit_transform(X_pca_scaled)
    
    # Analyze explained variance
    explained_variance_ratio = pca.explained_variance_ratio_
    cumulative_variance = np.cumsum(explained_variance_ratio)
    
    print(f"Explained variance by component:")
    for i, (var, cum_var) in enumerate(zip(explained_variance_ratio, cumulative_variance)):
        print(f"  PC{i+1}: {var:.4f} (cumulative: {cum_var:.4f})")
    
    # Find optimal number of components (95% variance)
    n_components_95 = np.argmax(cumulative_variance >= 0.95) + 1
    print(f"\nComponents needed for 95% variance: {n_components_95}")
    
    # Apply PCA with optimal components
    pca_optimal = PCA(n_components=n_components_95)
    X_pca_optimal = pca_optimal.fit_transform(X_pca_scaled)
    
    print(f"PCA-reduced features shape: {X_pca_optimal.shape}")
    
    # Train models with PCA features
    print("\nTraining models with PCA features...")
    
    # Split data
    X_train_pca, X_test_pca, y_train_pca, y_test_pca = train_test_split(
        X_pca_optimal, y_pca, test_size=0.2, random_state=42, stratify=y_pca
    )
    
    # Define models
    models_pca = {
        'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
        'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
        'Gradient Boosting': GradientBoostingClassifier(random_state=42),
        'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss'),
        'LightGBM': LGBMClassifier(random_state=42, verbose=-1)
    }
    
    # Train and evaluate models
    results_pca = {}
    
    for name, model in models_pca.items():
        print(f"\nTraining {name} with PCA features...")
        
        # Train model
        model.fit(X_train_pca, y_train_pca)
        
        # Make predictions
        y_pred_pca = model.predict(X_test_pca)
        y_pred_proba_pca = model.predict_proba(X_test_pca)[:, 1]
        
        # Calculate metrics
        accuracy = accuracy_score(y_test_pca, y_pred_pca)
        f1 = f1_score(y_test_pca, y_pred_pca)
        precision = precision_score(y_test_pca, y_pred_pca)
        recall = recall_score(y_test_pca, y_pred_pca)
        auc = roc_auc_score(y_test_pca, y_pred_proba_pca)
        
        # Store results
        results_pca[name] = {
            'accuracy': accuracy,
            'f1': f1,
            'precision': precision,
            'recall': recall,
            'auc': auc
        }
        
        print(f"  Accuracy: {accuracy:.4f}")
        print(f"  F1-score: {f1:.4f}")
        print(f"  Precision: {precision:.4f}")
        print(f"  Recall: {recall:.4f}")
        print(f"  AUC: {auc:.4f}")
    
    # Summary of PCA results
    print("\n" + "="*60)
    print("YBT PCA MODELS SUMMARY")
    print("="*60)
    
    results_pca_df = pd.DataFrame(results_pca).T
    results_pca_df = results_pca_df.sort_values('auc', ascending=False)
    
    print("\nPCA Results ranked by AUC:")
    print(results_pca_df.round(4))
    
    # Save PCA results
    results_pca_df.to_csv('data/processed/ybt_pca_models_results.csv')
    print(f"\nPCA results saved to: data/processed/ybt_pca_models_results.csv")

print("\n" + "="*80)
print("EXPERIMENT B COMPLETE (PCA ANALYSIS)")
print("="*80)


## 3. DESCRIPTIVE ANALYSIS & DATA VALIDATION

Before proceeding to additional experiments, let's conduct comprehensive descriptive analysis to validate our final dataset.


In [ ]:
print("="*80)
print("COMPREHENSIVE DESCRIPTIVE ANALYSIS OF FINAL YBT DATASET")
print("="*80)

# 1. Dataset Overview
print("1. DATASET OVERVIEW")
print("-" * 40)
print(f"Final dataset shape: {df.shape}")
print(f"Total samples: {len(df)}")
print(f"Total features: {len(df.columns)}")
print(f"Target distribution: {df['autism_target'].value_counts().to_dict()}")
print(f"Autism percentage: {df['autism_target'].mean()*100:.2f}%")

# 2. Questionnaire Score Distributions
print("\n2. QUESTIONNAIRE SCORE DISTRIBUTIONS")
print("-" * 40)
questionnaire_totals = ['eq_total', 'sqr_total', 'aq_total', 'd_score']
for col in questionnaire_totals:
    if col in df.columns:
        print(f"\n{col.upper()}:")
        print(f"  Range: {df[col].min():.1f} - {df[col].max():.1f}")
        print(f"  Mean: {df[col].mean():.2f} ± {df[col].std():.2f}")
        print(f"  Median: {df[col].median():.2f}")
        
        # By autism status
        autism_scores = df[df['autism_target']==1][col]
        non_autism_scores = df[df['autism_target']==0][col]
        print(f"  Autism cases: {autism_scores.mean():.2f} ± {autism_scores.std():.2f}")
        print(f"  Non-autism cases: {non_autism_scores.mean():.2f} ± {non_autism_scores.std():.2f}")

# 3. Clinical Validation
print("\n3. CLINICAL VALIDATION")
print("-" * 40)
if 'aq_total' in df.columns:
    # AQ clinical threshold analysis
    high_aq_cases = len(df[df['aq_total'] >= 6])
    autism_high_aq = len(df[(df['autism_target']==1) & (df['aq_total'] >= 6)])
    autism_low_aq = len(df[(df['autism_target']==1) & (df['aq_total'] < 6)])
    
    print(f"AQ Clinical Threshold (≥6):")
    print(f"  Total high AQ cases: {high_aq_cases} ({high_aq_cases/len(df)*100:.1f}%)")
    print(f"  Autism cases with high AQ: {autism_high_aq}")
    print(f"  Autism cases with low AQ: {autism_low_aq}")
    if autism_high_aq + autism_low_aq > 0:
        print(f"  Autism high AQ rate: {autism_high_aq/(autism_high_aq+autism_low_aq)*100:.1f}%")
    
    # Clinical interpretation
    print(f"\nClinical Interpretation:")
    autism_mean_aq = df[df['autism_target']==1]['aq_total'].mean()
    non_autism_mean_aq = df[df['autism_target']==0]['aq_total'].mean()
    print(f"  Autism cases mean AQ: {autism_mean_aq:.2f}")
    print(f"  Non-autism cases mean AQ: {non_autism_mean_aq:.2f}")
    print(f"  Clinical expectation: Autism cases should have HIGHER AQ scores")
    print(f"  Current finding: {'✅ CORRECT' if autism_mean_aq > non_autism_mean_aq else '❌ COUNTERINTUITIVE'}")

# 4. Feature Correlation Analysis
print("\n4. FEATURE CORRELATION ANALYSIS")
print("-" * 40)
# Get numeric features only
numeric_features = df.select_dtypes(include=[np.number]).columns
numeric_features = [col for col in numeric_features if col != 'autism_target']

# Calculate correlations with target
correlations = df[numeric_features].corrwith(df['autism_target']).abs().sort_values(ascending=False)
print("Top 10 features correlated with autism target:")
for i, (feature, corr) in enumerate(correlations.head(10).items()):
    print(f"  {i+1:2d}. {feature}: {corr:.4f}")

# 5. Data Quality Checks
print("\n5. DATA QUALITY CHECKS")
print("-" * 40)
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"Infinite values: {np.isinf(df.select_dtypes(include=[np.number])).sum().sum()}")

# Check for constant features
constant_features = []
for col in numeric_features:
    if df[col].nunique() <= 1:
        constant_features.append(col)
print(f"Constant features: {len(constant_features)} - {constant_features}")

print("\n" + "="*80)
print("DESCRIPTIVE ANALYSIS COMPLETE")
print("="*80)


# validation cell

In [ ]:
print("="*80)
print("FINAL VALIDATION: DATA LEAKAGE CHECK")
print("="*80)

# Check for any remaining AQ features
aq_features_check = [col for col in df.columns if 'aq' in col.lower()]
print(f"AQ features in final dataset: {aq_features_check}")

if len(aq_features_check) > 0:
    print("❌ DATA LEAKAGE DETECTED!")
    print("The following AQ features are still present:")
    for feature in aq_features_check:
        print(f"  - {feature}")
    print("\nThis will invalidate the model results!")
else:
    print("✅ No AQ features present - data leakage prevented")

# Check feature correlations with target
print("\nFeature correlations with autism target:")
numeric_features = df.select_dtypes(include=[np.number]).columns
numeric_features = [col for col in numeric_features if col != 'autism_target']

correlations = df[numeric_features].corrwith(df['autism_target']).abs().sort_values(ascending=False)
print("Top 5 feature correlations:")
for i, (feature, corr) in enumerate(correlations.head(5).items()):
    print(f"  {i+1}. {feature}: {corr:.4f}")

# Flag any suspiciously high correlations
high_corr_features = correlations[correlations > 0.5]
if len(high_corr_features) > 0:
    print(f"\n⚠️  WARNING: {len(high_corr_features)} features with very high correlation (>0.5):")
    for feature, corr in high_corr_features.items():
        print(f"  {feature}: {corr:.4f}")
else:
    print("\n✅ No suspiciously high correlations detected")

print("\n" + "="*80)
print("VALIDATION COMPLETE")
print("="*80)